<a href="https://colab.research.google.com/github/andysmokk/phyton_project/blob/main/a%7Cb_testanalysis_statistical_significance_key_metrics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [**Link to the Tableau visualization.**](https://public.tableau.com/app/profile/andrii.konovalov/viz/ABTest_17763507741560/Significance?publish=yes)

# [**Link to the Data File.**](https://drive.google.com/file/d/1XinkfY8saeSqGeHCVS2088LLJiKTvMpJ/view?usp=sharing)

In [ ]:
from google.colab import auth
from google.colab import drive
drive.mount('/content/drive')

from google.cloud import bigquery
from statsmodels.stats.proportion import proportions_ztest

import pandas as pd
import numpy as np

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
auth.authenticate_user()

In [ ]:
client = bigquery.Client(project="data-analytics-mate")

In [ ]:
query = """
WITH session_info AS (
  SELECT
    s.date,
    s.ga_session_id,
    sp.device,
    sp.continent,
    sp.country,
    sp.channel,
    ab.test,
    ab.test_group
  FROM
    `data-analytics-mate.DA.ab_test` ab
  JOIN
    `data-analytics-mate.DA.session` s
  ON
    ab.ga_session_id = s.ga_session_id
  JOIN
    `data-analytics-mate.DA.session_params` sp
  ON
    ab.ga_session_id = sp.ga_session_id
  ),


sessions_with_order AS (
  SELECT
    session_info.date,
    session_info.device,
    session_info.continent,
    session_info.country,
    session_info.channel,
    session_info.test,
    session_info.test_group,


    COUNT(DISTINCT o.ga_session_id) AS sessions_with_order
  FROM
    `data-analytics-mate.DA.order` o
  JOIN
    session_info
  ON
    o.ga_session_id = session_info.ga_session_id
  GROUP BY
    session_info.date,
    session_info.device,
    session_info.continent,
    session_info.country,
    session_info.channel,
    session_info.test,
    session_info.test_group
),


events AS (
  SELECT
    session_info.date,
    session_info.device,
    session_info.continent,
    session_info.country,
    session_info.channel,
    session_info.test,
    session_info.test_group,
    ep.event_name,


    COUNT(ep.ga_session_id) AS event_cnt
  FROM
    `data-analytics-mate.DA.event_params` ep
  JOIN
    session_info
  ON
    ep.ga_session_id = session_info.ga_session_id
  GROUP BY
    session_info.date,
    session_info.device,
    session_info.continent,
    session_info.country,
    session_info.channel,
    session_info.test,
    session_info.test_group,
    ep.event_name
),


sessions AS (
  SELECT
    session_info.date,
    session_info.device,
    session_info.continent,
    session_info.country,
    session_info.channel,
    session_info.test,
    session_info.test_group,


    COUNT(DISTINCT session_info.ga_session_id) AS sessions_cnt
  FROM
    session_info
  GROUP BY
    session_info.date,
    session_info.device,
    session_info.continent,
    session_info.country,
    session_info.channel,
    session_info.test,
    session_info.test_group
),


accounts AS (
  SELECT
    session_info.date,
    session_info.device,
    session_info.continent,
    session_info.country,
    session_info.channel,
    session_info.test,
    session_info.test_group,


    COUNT(DISTINCT acs.ga_session_id) AS new_accounts_cnt
  FROM
    `data-analytics-mate.DA.account_session` acs
  JOIN
    session_info
  ON
    acs.ga_session_id = session_info.ga_session_id
  GROUP BY
    session_info.date,
    session_info.device,
    session_info.continent,
    session_info.country,
    session_info.channel,
    session_info.test,
    session_info.test_group
)


SELECT
  sessions_with_order.date,
  sessions_with_order.device,
  sessions_with_order.continent,
  sessions_with_order.country,
  sessions_with_order.channel,
  sessions_with_order.test,
  sessions_with_order.test_group,
  'sessions with order' AS event_name,
  sessions_with_order.sessions_with_order AS value
FROM
  sessions_with_order
UNION ALL
SELECT
  events.date,
  events.device,
  events.continent,
  events.country,
  events.channel,
  events.test,
  events.test_group,
  events.event_name,
  events.event_cnt AS value
FROM
  events
UNION ALL
SELECT
  sessions.date,
  sessions.device,
  sessions.continent,
  sessions.country,
  sessions.channel,
  sessions.test,
  sessions.test_group,
  'sessions' AS event_name,
  sessions.sessions_cnt AS value
FROM
  sessions
UNION ALL
SELECT
  accounts.date,
  accounts.device,
  accounts.continent,
  accounts.country,
  accounts.channel,
  accounts.test,
  accounts.test_group,
  'new account' AS event_name,
  accounts.new_accounts_cnt AS value
FROM
  accounts;
"""

In [ ]:
query_job = client.query(query)  # Execute the SQL query.
results = query_job.result()  # Waiting for the query to finish.
df = results.to_dataframe()

In [ ]:
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800996 entries, 0 to 800995
Data columns (total 9 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   date        800996 non-null  dbdate
 1   device      800996 non-null  object
 2   continent   800996 non-null  object
 3   country     800996 non-null  object
 4   channel     800996 non-null  object
 5   test        800996 non-null  Int64 
 6   test_group  800996 non-null  Int64 
 7   event_name  800996 non-null  object
 8   value       800996 non-null  Int64 
dtypes: Int64(3), dbdate(1), object(5)
memory usage: 57.3+ MB


,test,test_group,value
count,800996.0,800996.0,800996.0
mean,2.831191,1.499608,9.475863
std,1.11694,0.5,37.493267
min,1.0,1.0,1.0
25%,2.0,1.0,1.0
50%,3.0,1.0,2.0
75%,4.0,2.0,5.0
max,4.0,2.0,1575.0


In [ ]:
df.head()

,date,device,continent,country,channel,test,test_group,event_name,value
0,2020-12-08,desktop,Asia,Palestine,Direct,4,2,new account,1
1,2020-12-08,desktop,Asia,Palestine,Direct,3,2,new account,1
2,2020-11-06,desktop,Americas,Puerto Rico,Social Search,2,2,new account,1
3,2020-11-06,desktop,Americas,Puerto Rico,Social Search,1,1,new account,1
4,2020-12-08,desktop,Europe,Croatia,Direct,4,2,new account,1


In [ ]:
df['event_name'].unique()


array(['new account', 'user_engagement', 'view_item', 'first_visit',
       'page_view', 'scroll', 'view_promotion', 'session_start',
       'view_search_results', 'add_shipping_info', 'select_item',
       'add_to_cart', 'select_promotion', 'begin_checkout',
       'add_payment_info', 'sessions with order', 'click', 'sessions',
       'view_item_list'], dtype=object)

In [ ]:
tested_metrics = ['add_payment_info', 'add_shipping_info', 'begin_checkout', 'new account']

df_tested_metrics = df[df['event_name'].isin(tested_metrics)]
df_tested_metrics.head()

,date,device,continent,country,channel,test,test_group,event_name,value
0,2020-12-08,desktop,Asia,Palestine,Direct,4,2,new account,1
1,2020-12-08,desktop,Asia,Palestine,Direct,3,2,new account,1
2,2020-11-06,desktop,Americas,Puerto Rico,Social Search,2,2,new account,1
3,2020-11-06,desktop,Americas,Puerto Rico,Social Search,1,1,new account,1
4,2020-12-08,desktop,Europe,Croatia,Direct,4,2,new account,1


In [ ]:
sessions = df_tested_metrics.groupby(['test', 'test_group', 'event_name'])['value'].sum().reset_index()
sessions.head()

,test,test_group,event_name,value
0,1,1,add_payment_info,1988
1,1,1,add_shipping_info,3034
2,1,1,begin_checkout,3784
3,1,1,new account,3823
4,1,2,add_payment_info,2229


In [ ]:
def ztest(df, test_number, metric_name):

    control_success = df.loc[(df['test_group'] == 1) & (df['event_name'] == metric_name), 'value'].sum()
    test_success = df.loc[(df['test_group'] == 2) & (df['event_name'] == metric_name), 'value'].sum()

    control_total_sessions = df.loc[(df['test_group'] == 1) & (df['event_name'] == 'sessions'), 'value'].sum()
    test_total_sessions = df.loc[(df['test_group'] == 2) & (df['event_name'] == 'sessions'), 'value'].sum()

    if control_total_sessions == 0 or test_total_sessions == 0:
        return None

    # successes = np.array([control_success, test_success])
    # nobs = np.array([control_total_sessions, test_total_sessions])

    successes = np.array([test_success, control_success])
    nobs = np.array([test_total_sessions, control_total_sessions])

    # Metric change
    p_control = control_success / control_total_sessions
    p_test = test_success / test_total_sessions
    metric_change = ((p_test - p_control) / p_control) * 100

    # z-test
    z_stat, p_value = proportions_ztest(successes, nobs)

    return {
        'test': test_number,
        'metric': metric_name,

        'numerator_event': metric_name,
        'denominator_event': 'sessions',

        'numerator_control': control_success,
        'denominator_control': control_total_sessions,
        'conversion_rate_control': p_control,

        'numerator_test': test_success,
        'denominator_test': test_total_sessions,
        'conversion_rate_test': p_test,

        'p-control': p_control,
        'p-test': p_test,

        'metric_change': metric_change,
        'z_stat': z_stat,
        'p_value': p_value,
        'significant': p_value < 0.05
        }

In [ ]:
results_total = []

for (test_number,), df_test in df.groupby(['test']):
    for metric in tested_metrics:
        result = ztest(df_test, test_number, metric)
        if result:
            results_total.append(result)

total_df = pd.DataFrame(results_total)
total_df

,test,metric,numerator_event,denominator_event,numerator_control,denominator_control,conversion_rate_control,numerator_test,denominator_test,conversion_rate_test,p-control,p-test,metric_change,z_stat,p_value,significant
0,1,add_payment_info,add_payment_info,sessions,1988,45362,0.043825,2229,45193,0.049322,0.043825,0.049322,12.542021,3.924884,0.000087,True
1,1,add_shipping_info,add_shipping_info,sessions,3034,45362,0.066884,3221,45193,0.071272,0.066884,0.071272,6.560481,2.603571,0.009226,True
2,1,begin_checkout,begin_checkout,sessions,3784,45362,0.083418,4021,45193,0.088974,0.083418,0.088974,6.660587,2.978783,0.002894,True
3,1,new account,new account,sessions,3823,45362,0.084278,3681,45193,0.081451,0.084278,0.081451,-3.354299,-1.542883,0.122859,False
4,2,add_payment_info,add_payment_info,sessions,2344,50637,0.046290,2409,50244,0.047946,0.046290,0.047946,3.576911,1.240994,0.214608,False
5,2,add_shipping_info,add_shipping_info,sessions,3480,50637,0.068724,3510,50244,0.069859,0.068724,0.069859,1.650995,0.709557,0.477979,False
6,2,begin_checkout,begin_checkout,sessions,4262,50637,0.084168,4313,50244,0.085841,0.084168,0.085841,1.988164,0.952898,0.340642,False
7,2,new account,new account,sessions,4165,50637,0.082252,4184,50244,0.083274,0.082252,0.083274,1.241934,0.588793,0.556000,False
8,3,add_payment_info,add_payment_info,sessions,3623,70047,0.051722,3697,70439,0.052485,0.051722,0.052485,1.474630,0.643172,0.520112,False
9,3,add_shipping_info,add_shipping_info,sessions,5298,70047,0.075635,5188,70439,0.073652,0.075635,0.073652,-2.621211,-1.413727,0.157442,False


In [ ]:
# total_df.to_csv('/content/drive/MyDrive/MATE_ACADEMY/COLAB/portfolio_projects/ab_test_total.csv', sep=';', index=False)

In [ ]:
results_device_channel = []

for (test_number, device, channel), df_group in df.groupby(['test', 'device', 'channel']):
    for metric in tested_metrics:
        result = ztest(df_group, test_number, metric)
        if result:
            result['device'] = device
            result['channel'] = channel
            results_device_channel.append(result)

device_channel_df = pd.DataFrame(results_device_channel)
device_channel_df

,test,metric,numerator_event,denominator_event,numerator_control,denominator_control,conversion_rate_control,numerator_test,denominator_test,conversion_rate_test,p-control,p-test,metric_change,z_stat,p_value,significant,device,channel
0,1,add_payment_info,add_payment_info,sessions,227,6258,0.036274,275,6044,0.045500,0.036274,0.045500,25.434771,2.585789,9.715631e-03,True,desktop,Direct
1,1,add_shipping_info,add_shipping_info,sessions,375,6258,0.059923,421,6044,0.069656,0.059923,0.069656,16.241694,2.193695,2.825736e-02,True,desktop,Direct
2,1,begin_checkout,begin_checkout,sessions,462,6258,0.073826,553,6044,0.091496,0.073826,0.091496,23.935082,3.561123,3.692726e-04,True,desktop,Direct
3,1,new account,new account,sessions,541,6258,0.086449,489,6044,0.080907,0.086449,0.080907,-6.411455,-1.109602,2.671708e-01,False,desktop,Direct
4,1,add_payment_info,add_payment_info,sessions,334,9221,0.036222,271,9089,0.029816,0.036222,0.029816,-17.683908,-2.424445,1.533182e-02,True,desktop,Organic Search
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
235,4,new account,new account,sessions,20,173,0.115607,17,188,0.090426,0.115607,0.090426,-21.781915,-0.788065,4.306586e-01,False,tablet,Social Search
236,4,add_payment_info,add_payment_info,sessions,16,126,0.126984,4,120,0.033333,0.126984,0.033333,-73.750000,-2.686493,7.220649e-03,True,tablet,Undefined
237,4,add_shipping_info,add_shipping_info,sessions,15,126,0.119048,4,120,0.033333,0.119048,0.033333,-72.000000,-2.517137,1.183127e-02,True,tablet,Undefined
238,4,begin_checkout,begin_checkout,sessions,42,126,0.333333,9,120,0.075000,0.333333,0.075000,-77.500000,-4.995989,5.853489e-07,True,tablet,Undefined


In [ ]:
# device_channel_df.to_csv('/content/drive/MyDrive/MATE_ACADEMY/COLAB/portfolio_projects/ab_test_device_channel.csv', sep=';', index=False)

In [ ]:
results_device = []

for (test_number, device), df_group in df.groupby(['test', 'device']):
    for metric in tested_metrics:
        result = ztest(df_group, test_number, metric)
        if result:
            result['device'] = device
            results_device.append(result)

device_df = pd.DataFrame(results_device)
device_df

,test,metric,numerator_event,denominator_event,numerator_control,denominator_control,conversion_rate_control,numerator_test,denominator_test,conversion_rate_test,p-control,p-test,metric_change,z_stat,p_value,significant,device
0,1,add_payment_info,add_payment_info,sessions,1130,26467,0.042695,1256,26417,0.047545,0.042695,0.047545,11.360819,2.686998,0.007210,True,desktop
1,1,add_shipping_info,add_shipping_info,sessions,1711,26467,0.064647,1916,26417,0.072529,0.064647,0.072529,12.193247,3.586024,0.000336,True,desktop
2,1,begin_checkout,begin_checkout,sessions,2108,26467,0.079646,2404,26417,0.091002,0.079646,0.091002,14.257595,4.673980,0.000003,True,desktop
3,1,new account,new account,sessions,2203,26467,0.083236,2147,26417,0.081273,0.083236,0.081273,-2.357527,-0.821212,0.411526,False,desktop
4,1,add_payment_info,add_payment_info,sessions,810,17896,0.045262,942,17767,0.053020,0.045262,0.053020,17.140683,3.389330,0.000701,True,mobile
5,1,add_shipping_info,add_shipping_info,sessions,1257,17896,0.070239,1256,17767,0.070693,0.070239,0.070693,0.645933,0.167387,0.867065,False,mobile
6,1,begin_checkout,begin_checkout,sessions,1593,17896,0.089014,1561,17767,0.087860,0.089014,0.087860,-1.297308,-0.384029,0.700957,False,mobile
7,1,new account,new account,sessions,1530,17896,0.085494,1441,17767,0.081105,0.085494,0.081105,-5.133163,-1.499486,0.133748,False,mobile
8,1,add_payment_info,add_payment_info,sessions,48,999,0.048048,31,1009,0.030723,0.048048,0.030723,-36.056739,-1.996608,0.045868,True,tablet
9,1,add_shipping_info,add_shipping_info,sessions,66,999,0.066066,49,1009,0.048563,0.066066,0.048563,-26.493378,-1.687725,0.091464,False,tablet


In [ ]:
# device_df.to_csv('/content/drive/MyDrive/MATE_ACADEMY/COLAB/portfolio_projects/ab_test_device.csv', sep=';', index=False)

In [ ]:
results_channel = []

for (test_number, channel), df_group in df.groupby(['test', 'channel']):
    for metric in tested_metrics:
        result = ztest(df_group, test_number, metric)
        if result:
            result['channel'] = channel
            results_channel.append(result)

channel_df = pd.DataFrame(results_channel)
channel_df

,test,metric,numerator_event,denominator_event,numerator_control,denominator_control,conversion_rate_control,numerator_test,denominator_test,conversion_rate_test,p-control,p-test,metric_change,z_stat,p_value,significant,channel
0,1,add_payment_info,add_payment_info,sessions,392,10691,0.036666,516,10361,0.049802,0.036666,0.049802,35.825180,4.690261,0.000003,True,Direct
1,1,add_shipping_info,add_shipping_info,sessions,664,10691,0.062108,716,10361,0.069105,0.062108,0.069105,11.265775,2.050707,0.040295,True,Direct
2,1,begin_checkout,begin_checkout,sessions,823,10691,0.076981,915,10361,0.088312,0.076981,0.088312,14.719677,2.986589,0.002821,True,Direct
3,1,new account,new account,sessions,913,10691,0.085399,850,10361,0.082038,0.085399,0.082038,-3.935085,-0.879999,0.378860,False,Direct
4,1,add_payment_info,add_payment_info,sessions,640,15675,0.040829,514,15631,0.032883,0.040829,0.032883,-19.461427,-3.730758,0.000191,True,Organic Search
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,4,new account,new account,sessions,667,7961,0.083783,653,8056,0.081058,0.083783,0.081058,-3.253444,-0.627241,0.530501,False,Social Search
76,4,add_payment_info,add_payment_info,sessions,496,5716,0.086774,566,5862,0.096554,0.086774,0.096554,11.270787,1.822812,0.068332,False,Undefined
77,4,add_shipping_info,add_shipping_info,sessions,640,5716,0.111966,666,5862,0.113613,0.111966,0.113613,1.470701,0.280026,0.779457,False,Undefined
78,4,begin_checkout,begin_checkout,sessions,1578,5716,0.276067,1600,5862,0.272944,0.276067,0.272944,-1.131171,-0.376454,0.706579,False,Undefined


In [ ]:
# channel_df.to_csv('/content/drive/MyDrive/MATE_ACADEMY/COLAB/portfolio_projects/ab_test_channel.csv', sep=';', index=False)

# **Overwiev of the A|B test**

## **1. Significance In Total.**

**The overall result showed the ineffectiveness of the test group**, as statistical significance for three out of four key metrics was observed only in the first test. Specifically, there was an increase in conversion rates: `add_payment_info / session by 4.93%`, `add_shipping_info / session by 7.13%`, and `begin_checkout / session by 8.9%`. In relative terms, these metrics improved by 12.54%, 6.56%, and 6.66%, respectively.

**A slight decline was observed only in the** `new_accounts / session` metric, but **it can be ignored since the result is not statistically significant**. Therefore, it is most likely due to random fluctuations.

**As for tests 2, 3, and 4, the metrics showed either a decrease or a lack of statistical significance.**

## **2. Significance By Device.**

The results broken down by device differ somewhat from the overall results. In particular, **Test 1 showed an increase in conversion across all key metrics on desktop**, as well as for `add_payment_info / session` on `mobile`. However, `add_payment_info / session` `and begin_checkout / session` on `tablet` showed a decline in `metric change`. The `begin_checkout / session` metric also showed a decrease, but without statistical significance.

**Tests 2 and 3 showed either a decline in almost all metrics or a lack of statistical significance.**

**Test 4 demonstrated a statistically significant increase in conversion across all metrics on `desktop`**, but a decline in metric change compared to the control group. Metrics on `mobile` were not statistically significant, except for `begin_checkout / session`, which showed a significant increase. `tablet` showed a slight but statistically significant increase in conversion rates, but a substantial decline in metric change.

## **3. Significance By Channel.**

**The test results by `traffic sources` did not show any significant** overall improvement for the test group. There were minor improvements in certain key metrics for specific sources, but they lack statistical significance or are unlikely to impact the overall result, and therefore should not be taken into account.

## **4. Significance By Device And Channel.**

**The results broken down simultaneously by `device` and `traffic source` also do not show any significant improvement in key metrics**, which further confirms the previous conclusions about the lack of effectiveness of the test variant, taking into account the analysis across individual segments by device and traffic source.

# **Conclusions and decisions**

**At this stage, I recommend not implementing the changes**, as they are unlikely to lead to improvements in business metrics. Additionally, **it would be advisable to further analyze specific elements that may be useful for future updates or are easier to maintain**, and to conduct additional research before making a final decision.